# Resolution Matrix

Let the following define the forward problem : $$\mathbf y = \mathbf{Ls}$$ Where $\mathbf y$ is the M/EEG data, $\mathbf s$ are the source activity and $\mathbf L$ is the lead field matrix. <br><br>
If the inverse problem is linear we can define a linear estimator matrix $\mathbf G$ as: $$\mathbf{\hat s} = \mathbf{Gy}$$
<br><br>
From the above equations follows that $$\mathbf{\hat s} = \mathbf{Gy} = \mathbf{GLs} = \mathbf{Rs}$$ $\mathbf R$ is called the "resolution matrix". The resolution matrix contains a lot of information about the spatial resoluion of linear estimators, by means of their point-spread and cross-talk functions (PSFs and CTFs). <br><br>

In this notebook, we will compute the resolution matrix for various methods and compare their point-spread function as well as their cross-talk functions to evaluate their performance.

## 📦 Imports

In [13]:
import mindmap as mm
import mne

## 🧠 Data

📥 **<font color=pink>Inputs</font>**
- fwd: Forward solution
- eeg.info: EEG informations
- noise_cov: EEG noise covariance matrix

In [14]:
# ------------ Parameters -------------
fconfig = "../../papers/simf/configuration.yml"
sub     = "10"
task    = "rest"
run     = 11
# -------------------------------------

reader = mm.io.dataset_reader(fconfig)
data   = reader(sub, "ave", "fwd", "ncov", "eig", task=task, run=run)

DATA SUMMARY
Dataset: simf
Subject: 10
Task: rest
Run: 11
Retrieved objects:
- ncov
- eig
- fwd
- ave
Missing objects:


## ⚙️ Build inverse

**Available methods** (only linear inverse operators): 

- <font color=gray>MNE</font>: Assumes independent gaussian distributed sources. Uses depth weighting to correct for depth bias.
- <font color=gray>sLORETA</font>: Assumes independent gaussian distributed sources. Applies a post-hoc standardization.
- <font color=gray>dSMP</font>: ...
- <font color=gray>beamformer</font>: ...
- <font color=gray>wCSET</font>: Assumes independent gaussian distributed connectome spectrum with components weighted by spatial frequency.

🎚️ **<font color=pink>Inverse</font>**:
- `method`: ["MNE", "sLORETA", "dSMP", "beamformer", "CSET", "wCSET"]
- `lam`: Regularizer parameter
- `depth`: Depth weighting exponent
- `relax`: Harmonic weight relaxation

In [39]:
# ------------ Parameters -------------
# method
method = "sloreta"

# sloreta parameters
lam = 0.01

# wcset parameters
harm      = 0.7
lam_harm  = 0.1

depth     = 0.5
lam_depth = 0.1

# -------------------------------------

if method.lower() == "sloreta":
    ncov = mne.make_ad_hoc_cov(data["ave"].info, std={"eeg":1})
    inverse = mne.minimum_norm.make_inverse_operator(
        info      = data["ave"].info,
        forward   = data["fwd"],
        noise_cov = ncov,
        fixed     = True,
        depth     = depth,               # depth weighting doesn't do anything in sLORETA
        verbose   = "warning",
    )

elif method.lower() == "wcset":
    # compute wCSET solution
    inverse = mm.inverse.wCSET(
        info            = data["ave"].info,
        leadfield       = data["fwd"],
        noise_cov       = None,
        eig_vec         = data["eig"]["vec"],
        eig_val         = data["eig"]["val"],
        depth           = depth,
        harm            = harm,
    )

else:
    raise ValueError(f"Unknown method '{method}'")

## ⚙️ Resolution Matrix

#### Point spread function
`psf`: A point spread function (PSF) characterizes how an imaging system transforms an ideal point source into a spatially distributed signal, effectively describing the system's blurring behavior. In this context, column $i$ of the resolution matrix represents the point spread associated with dipole $i$.

#### Cross talk function
`ctf`: The cross talk function (CTF) for a given source location describes how activity at other source locations can influence the estimate at that location. In other words, it quantifies the degree of "contamination" from surrounding sources. The $i$-th row of the resolution matrix corresponds to the CTF for location $i$.

#### Metrics
- `peak_err`: Peak localization error (PLE), Euclidean distance between peak and true source location. 
- `cog_err`: Centre-of-gravity localisation error (CoG), Euclidean distance between CoG and true source location.
- `sd_ext`: Spatial deviation.
- `maxrad_ext`: Maximum radius to 50% of max amplitude.
- `peak_amp`: Ratio between absolute maximum amplitudes of peaks per location and maximum peak across locations.
- `sum_amp`: Ratio between sums of absolute amplitudes.

In [40]:
# fix dipole orientation
forward = mne.convert_forward_solution(data["fwd"], force_fixed=True, verbose="warning")

# compute resolution matrix
if method.lower() in ["mne", "sloreta", "dspm", "beamformer"]:
    resmat = mne.minimum_norm.make_inverse_resolution_matrix(
        forward          = forward, 
        inverse_operator = inverse, 
        method           = "sLORETA", 
        lambda2          = 0.01, 
        verbose          = "warning",
    )
else:
    resmat = inverse.make_resolution_matrix(lam_harm, lam_depth)
    
# compute resolution metrics 
resolution_metrics = {"psf":{}, "ctf":{}} 
metrics            = ["peak_err", "cog_err", "sd_ext", "maxrad_ext", "peak_amp", "sum_amp"]

for f in resolution_metrics.keys():
    for m in metrics:
        resolution_metrics[f][m] = mne.minimum_norm.resolution_metrics(
            resmat   = resmat,
            src      = data["fwd"]["src"],
            function = f,
            metric   = m,
            verbose  = "warning",
        )

📊 **Point spread and cross talk functions**

- `func`: type of fonction ("psf" or "ctf")

In [41]:
########### Parameters ###########
func = "ctf"
##################################

if func == "psf":
    rm = resmat.copy()
elif func == "ctf":
    rm = resmat.T.copy()
else:
    raise ValueError(f"Unknown function name: '{func}'")

mm.viz.plot_dipole_function(data["fwd"]["src"], rm).show()

Widget(value='<iframe src="http://localhost:35385/index.html?ui=P_0x7824228cb350_20&reconnect=auto" class="pyv…

📊 **Function metrics**

- `func`: type of function ("psf" or "ctf")
- `metric`: type of metric ("peak_err", "cog_err", "sd_ext", "maxrad_ext", "peak_amp", "sum_amp")

In [43]:
############# Parameters ##############
func = "psf"
metric = "peak_err"
#######################################

signals = resolution_metrics[func][metric].data.squeeze()
mm.viz.plot_brain(data["fwd"]["src"], [signals,signals,signals,signals], title=f"{func}:{metric}").show()

Widget(value='<iframe src="http://localhost:35385/index.html?ui=P_0x7824b47d69d0_22&reconnect=auto" class="pyv…

## Harmonic resolution

In [7]:
harmat = inverse.make_harmonic_resolution_matrix(data["ave"])

In [8]:
test = data["eig"]["vec"].T @ resmat
mm.viz.plot_spectrum(test[:,:300])

No times provided


In [3]:
from pathlib import Path
test = Path(reader.config["MNI"])

In [10]:
mm.io.check_file(test.suffix, extension=["mgz"])

FileNotFoundError: Following file was not found: .nii

In [12]:
test.name.replace(".nii", ".mgz")
print(test)

/media/nino/DATA/templates/mni_icbm152_nlin_asym_09c_nifti/mni_icbm152_nlin_asym_09c/mni_icbm152_t1_tal_nlin_asym_09c.nii
